In [1]:
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import NearestNeighbors

In [3]:
articles = pd.read_parquet("../../../data/gold/articles.parquet")

articles.shape

(105542, 14)

In [4]:
categorical_cols = [
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "perceived_colour_master_name",
    "department_name",
    "index_name",
    "index_group_name",
    "section_name",
    "garment_group_name"
]

In [5]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

categorical_matrix = encoder.fit_transform(articles[categorical_cols])

categorical_matrix.shape

(105542, 600)

In [6]:
nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn_model.fit(categorical_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [7]:
query_idx = 0

distances, indices = nn_model.kneighbors(
    categorical_matrix[query_idx],
    n_neighbors=11
)

distances, indices

(array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
 array([[     0, 102303,  76305, 101064, 102963,   5317, 101344,  77177,
          77092,  35623,  72486]]))

In [8]:
query_idx = 0

neighbor_indices = indices[0][1:]
neighbor_distances = distances[0][1:]

display_cols = [
    "article_id",
    "prod_name",
    "product_type_name",
    "colour_group_name",
    "section_name",
    "garment_group_name"
]

query_product = articles.iloc[[query_idx]][display_cols]

recommendations = articles.iloc[neighbor_indices][display_cols].copy()
recommendations["cosine_similarity"] = 1 - neighbor_distances

query_product, recommendations

(   article_id  prod_name product_type_name colour_group_name  \
 0   108775015  Strap top          Vest top             Black   
 
              section_name garment_group_name  
 0  Womens Everyday Basics       Jersey Basic  ,
         article_id                   prod_name product_type_name  \
 102303   903309001         V-neck straptop 3-p          Vest top   
 76305    784338001              Petra seamless          Vest top   
 101064   893994001  V-Neck Strap Top Long 2-pk          Vest top   
 102963   908230001         Moa rib 2-pack tank          Vest top   
 5317     494030013                    Tika (1)          Vest top   
 101344   895993001                    Dag Tank          Vest top   
 77177    788261001                 Amanda Tank          Vest top   
 77092    787946002             Moa 2 pack tank          Vest top   
 35623    647505001                  Perth tank          Vest top   
 72486    767834001                  Strap Top.          Vest top   
 
        co

In [12]:
def get_similar_products(article_id, top_k=10):
    matching_indices = articles.index[
        articles["article_id"] == article_id
    ].tolist()

    if not matching_indices:
        raise ValueError(f"article_id bulunamadı: {article_id}")

    query_idx = matching_indices[0]

    # Biraz fazla komşu isteyelim ki kendisini çıkardıktan sonra top_k ürün kalsın
    distances, indices = nn_model.kneighbors(
        categorical_matrix[query_idx],
        n_neighbors=top_k + 5
    )

    neighbor_indices = indices[0]
    neighbor_distances = distances[0]

    # Sorgu ürününün kendisini açıkça çıkar
    filtered_neighbors = [
        (idx, dist)
        for idx, dist in zip(neighbor_indices, neighbor_distances)
        if idx != query_idx
    ]

    # İlk top_k sonucu al
    filtered_neighbors = filtered_neighbors[:top_k]

    neighbor_indices = [idx for idx, _ in filtered_neighbors]
    neighbor_distances = [dist for _, dist in filtered_neighbors]

    display_cols = [
        "article_id",
        "prod_name",
        "product_type_name",
        "colour_group_name",
        "section_name",
        "garment_group_name"
    ]

    query_product = articles.iloc[[query_idx]][display_cols]

    recommendations = articles.iloc[neighbor_indices][display_cols].copy()
    recommendations["cosine_similarity"] = 1 - pd.Series(
        neighbor_distances,
        index=recommendations.index
    )

    return query_product, recommendations

In [10]:
query_product, recommendations = get_similar_products(
    article_id=108775015,
    top_k=10
)

query_product, recommendations

(   article_id  prod_name product_type_name colour_group_name  \
 0   108775015  Strap top          Vest top             Black   
 
              section_name garment_group_name  
 0  Womens Everyday Basics       Jersey Basic  ,
         article_id                   prod_name product_type_name  \
 102303   903309001         V-neck straptop 3-p          Vest top   
 76305    784338001              Petra seamless          Vest top   
 101064   893994001  V-Neck Strap Top Long 2-pk          Vest top   
 102963   908230001         Moa rib 2-pack tank          Vest top   
 5317     494030013                    Tika (1)          Vest top   
 101344   895993001                    Dag Tank          Vest top   
 77177    788261001                 Amanda Tank          Vest top   
 77092    787946002             Moa 2 pack tank          Vest top   
 35623    647505001                  Perth tank          Vest top   
 72486    767834001                  Strap Top.          Vest top   
 
        co

In [13]:
test_article_ids = [
    108775015,  # Strap top
    681107007,  # Dress
    686284001,  # Sweater
    754256001,  # Bra
    755712001   # Shirt
]

for article_id in test_article_ids:
    query_product, recommendations = get_similar_products(
        article_id=article_id,
        top_k=5
    )

    print("\nQUERY PRODUCT")
    display(query_product)

    print("RECOMMENDATIONS")
    display(recommendations)


QUERY PRODUCT


,article_id,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
0,108775015,Strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic


RECOMMENDATIONS


,article_id,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
102303,903309001,V-neck straptop 3-p,Vest top,Black,Womens Everyday Basics,Jersey Basic,1.0
76305,784338001,Petra seamless,Vest top,Black,Womens Everyday Basics,Jersey Basic,1.0
101064,893994001,V-Neck Strap Top Long 2-pk,Vest top,Black,Womens Everyday Basics,Jersey Basic,1.0
102963,908230001,Moa rib 2-pack tank,Vest top,Black,Womens Everyday Basics,Jersey Basic,1.0
5317,494030013,Tika (1),Vest top,Black,Womens Everyday Basics,Jersey Basic,1.0



QUERY PRODUCT


...

,article_id,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
17183,573150001,KARIN fancy+,Bra,Black,Ladies H&M Sport,Jersey Fancy,1.0
61175,732413001,PANORAMA sports bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,1.0
93493,858407002,Solo Assymetric bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,1.0
32252,634320003,PANORAMA sports bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,1.0
63464,739347001,KARIN MID SUPPORT BRA,Bra,Black,Ladies H&M Sport,Jersey Fancy,1.0



QUERY PRODUCT


,article_id,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
68888,755712001,Sune slipover set,Shirt,White,Kids Boy,Shirts


RECOMMENDATIONS


,article_id,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
83723,813060015,Princeton ls shirt (TVP),Shirt,White,Kids Boy,Shirts,1.0
83961,814200001,Steve collarless,Shirt,White,Kids Boy,Shirts,1.0
31508,632376003,Kent ss shirt,Shirt,White,Kids Boy,Shirts,1.0
60098,726726001,TP Princeton shirt,Shirt,White,Kids Boy,Shirts,1.0
21079,592039001,Penguin shirt 2SET,Shirt,White,Kids Boy,Shirts,1.0


In [14]:
signature_group_sizes = (
    articles
    .groupby(categorical_cols, observed=True)
    .size()
    .reset_index(name="group_size")
)

signature_group_sizes["group_size"].describe()

count    42882.000000
mean         2.461219
std          5.929257
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        321.000000
Name: group_size, dtype: float64

In [15]:
{
    "unique_signature_groups": len(signature_group_sizes),
    "products_in_non_unique_groups": signature_group_sizes.loc[
        signature_group_sizes["group_size"] > 1, "group_size"
    ].sum(),
    "max_same_signature_group_size": signature_group_sizes["group_size"].max()
}

{'unique_signature_groups': 42882,
 'products_in_non_unique_groups': np.int64(77870),
 'max_same_signature_group_size': np.int64(321)}

## Initial Findings

The categorical similarity baseline successfully retrieves products from highly relevant product families.  
For example, products with the same product type, colour, section, and garment group are often ranked at the top.

However, the analysis also shows an important limitation:

- 42,882 unique categorical signatures were found among 105,542 articles.
- 77,870 products share their categorical signature with at least one other product.
- The largest identical-signature group contains 321 products.

This indicates that categorical metadata is useful for narrowing down candidate products, but it is often insufficient to rank very similar items within the same category group.

A text-based representation using `prod_name` and `detail_desc` should be explored next to improve fine-grained similarity ranking.